In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Wazirpur,Delhi - DPCC.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,Toluene,RH,WS,WD,SR,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,189.49,281.73,25.90,62.87,54.52,1.52,15.48,5.35,5.67,83.05,1.67,251.13,82.20,980.16,12.28,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,222.59,323.21,36.88,63.84,63.91,2.62,54.92,6.49,7.02,85.72,1.54,214.23,70.51,980.18,12.38,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,327.39,505.64,115.75,97.09,145.73,2.87,71.20,10.74,8.76,87.25,0.97,221.62,77.51,981.24,13.44,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,343.04,500.92,85.37,119.47,132.90,4.17,44.25,13.29,10.54,83.17,1.82,201.79,73.31,980.10,13.48,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,223.00,346.42,36.04,82.17,72.98,1.72,38.72,8.39,4.80,83.35,2.45,157.34,87.74,979.29,13.28,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,401.53,681.98,26.77,75.63,61.94,2.31,49.78,1.90,13.51,51.69,1.19,213.70,73.36,969.00,18.53,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,356.46,617.83,17.26,84.20,58.81,2.42,51.66,2.21,21.51,52.19,1.10,195.68,67.31,969.00,18.53,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,276.49,539.98,16.06,86.11,59.17,1.71,49.49,2.16,17.78,51.98,1.05,184.78,67.34,969.00,18.35,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,299.50,539.88,23.24,85.77,64.52,1.78,50.83,2.33,14.23,51.51,1.08,198.93,68.76,969.00,18.28,0.0,0.0


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 19)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): []
Dropped rows (>70% NaN): 0
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
CO           0
Ozone        0
Benzene      0
Toluene      0
RH           0
WS           0
WD           0
SR           0
BP           0
AT           0
RF           0
TOT-RF       0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:

# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (320, 19)
          From Date           To Date    PM2.5    PM10     NO     NO2    NOx  \
0  01-01-2025 00:00  02-01-2025 00:00  189.490  281.73  25.90   62.87  54.52   
1  02-01-2025 00:00  03-01-2025 00:00  222.590  323.21  36.88   63.84  63.91   
2  03-01-2025 00:00  04-01-2025 00:00   74.125  505.64  15.50   97.09  43.95   
3  04-01-2025 00:00  05-01-2025 00:00   74.125  500.92  15.50  119.47  43.95   
4  05-01-2025 00:00  06-01-2025 00:00  223.000  346.42  36.04   82.17  72.98   

     CO  Ozone  Benzene  Toluene     RH    WS      WD     SR      BP     AT  \
0  1.52  15.48    5.350     5.67  83.05  1.67  251.13  82.20  980.16  12.28   
1  1.12  54.92    6.490     7.02  85.72  1.54  214.23  70.51  980.18  12.38   
2  1.12  71.20    0.885     8.76  87.25  0.97  221.62  77.51  981.24  13.44   
3  1.12  44.25    0.885    10.54  83.17  1.82  201.79  73.31  980.10  13.48   
4  1.72  38.72    8.390     4.80  83.35  2.45  157.34  87.74  979.29  13.28   

    RF  TOT-RF  
0  0

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,Toluene,RH,WS,WD,SR,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,2.191148,0.333237,0.888410,-0.008620,0.219533,0.914459,-1.450074,1.430319,0.324200,1.959389,-0.399445,0.787725,-0.083059,0.957532,-2.103254,-0.346616,-0.355982
1,02-01-2025 00:00,03-01-2025 00:00,2.876499,0.709919,2.025844,0.024092,0.612400,-0.001646,0.197422,1.918370,0.788417,2.185827,-0.601082,0.022226,-0.481394,0.958817,-2.087225,-0.346616,-0.355982
2,03-01-2025 00:00,04-01-2025 00:00,-0.197536,2.366578,-0.188941,1.145400,-0.222703,-0.001646,0.877474,-0.481215,1.386742,2.315583,-1.485183,0.175533,-0.242870,1.026935,-1.917322,-0.346616,-0.355982
3,04-01-2025 00:00,05-01-2025 00:00,-0.197536,2.323715,-0.188941,1.900132,-0.222703,-0.001646,-0.248288,-0.481215,1.998821,1.969566,-0.166787,-0.235845,-0.385984,0.953676,-1.910910,-0.346616,-0.355982
4,05-01-2025 00:00,06-01-2025 00:00,2.884988,0.920691,1.938827,0.642245,0.991878,1.372512,-0.479288,2.731789,0.025038,1.984832,0.810377,-1.157971,0.105715,0.901624,-1.942968,-0.346616,-0.355982
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,-0.197536,-0.148922,0.978534,0.421693,0.529977,2.723767,-0.017287,-0.046678,3.020099,-0.700191,-1.143951,0.011231,-0.384281,0.240364,-1.101466,-0.346616,-0.355982
316,13-11-2025 00:00,14-11-2025 00:00,-0.197536,-0.148922,-0.006620,0.710703,0.399022,2.975696,0.061245,0.086038,-0.262089,-0.657787,-1.283546,-0.362598,-0.590433,0.240364,-1.101466,-0.346616,-0.355982
317,14-11-2025 00:00,15-11-2025 00:00,-0.197536,2.678422,-0.130930,0.775115,0.414084,1.349609,-0.029401,0.064632,-0.262089,-0.675597,-1.361099,-0.588721,-0.589411,0.240364,-1.130317,-0.346616,-0.355982
318,15-11-2025 00:00,16-11-2025 00:00,-0.197536,2.677514,0.612857,0.763649,0.637921,1.509928,0.026574,0.137412,3.267682,-0.715457,-1.314567,-0.295176,-0.541025,0.240364,-1.141537,-0.346616,-0.355982


In [10]:
df.to_excel('wazirpur2025.xlsx', index=False)